# Information Health — Metric Validation (developer / QA)

Independently recomputes **every Information Health metric** from a Reading History and checks it
against the **unchanged** engine — the raw layer *and* the percentile (displayed) layer — then reports
**PASS / FAIL**, drift, and charts.

**This is not the product.** It is a developer / research / QA tool. By default it runs **completely
offline** through the [Metric Validation Pipeline](../docs/METRIC_PIPELINE.md) — no dashboard, web app,
browser extension, or engine required. It only consumes a **Reading History** (golden personas, a JSON
file, or a stored user) and the pipeline itself.

An **optional** *Dashboard Verification* mode (last section) additionally compares the independently
calculated metrics against a **running application's** Dashboard/API values — only when you point it at
a live engine. The default workflow never needs one.

Run the cells top to bottom (Runtime → *Run all* also works).

In [ ]:
#@title 1 · Setup — clone the repo + install (pure Python; no Node / engine)
import os, sys, subprocess, pathlib

REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}         # only if the repo is private

def _in_repo():
    return pathlib.Path("examples/metric_pipeline").is_dir()

# Colab: clone fresh. Already inside the repo (e.g. a CI run): use it in place.
if not _in_repo():
    auth = (GITHUB_TOKEN + "@") if GITHUB_TOKEN else ""
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://%sgithub.com/%s.git" % (auth, REPO), "app"], check=True)
    os.chdir("app")
REPO_ROOT = os.getcwd()
print("repo:", REPO_ROOT)

# The pipeline needs numpy/scipy/pandas (installed with the rwe package); matplotlib for the charts;
# sqlalchemy only for the optional stored-user source. None of the product's Node/web/engine stack.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib", "sqlalchemy"], check=True)

# Import the pipeline in-process too (charts / Dashboard Verification reuse its stages).
sys.path.insert(0, os.path.join(REPO_ROOT, "examples"))
import metric_pipeline  # noqa: F401  (also puts the repo root on sys.path for the engine it checks)
print("ready · python", sys.version.split()[0])

In [ ]:
#@title 2 · Choose the validation source
# Offline by default. "golden" needs nothing; "history" reads a JSON file; "user" reads a stored
# reading history (seeded into a throwaway local DB in the next cell if empty).
SOURCE       = "golden"  #@param ["golden", "user", "history"]
PERSONA      = "all"     #@param ["all", "balanced", "echo_chamber", "opinion_heavy", "technology", "single_publisher", "global_reader"]
USER         = "1"       #@param {type:"string"}
HISTORY_FILE = ""        #@param {type:"string"}

if SOURCE == "golden":
    ARGS = ["--golden", PERSONA or "all"]
    DATASET_DESC = "golden persona(s): %s" % (PERSONA or "all")
elif SOURCE == "user":
    ARGS = ["--user", str(USER)]
    DATASET_DESC = "stored user #%s" % USER
elif SOURCE == "history":
    assert HISTORY_FILE, "set HISTORY_FILE to a reads JSON path"
    ARGS = ["--history", HISTORY_FILE]
    DATASET_DESC = "reading-history file: %s" % HISTORY_FILE
else:
    raise ValueError("SOURCE must be golden | user | history")
print("source →", DATASET_DESC)
print("pipeline args →", " ".join(ARGS))

In [ ]:
#@title 2b · (only for SOURCE = "user") seed a throwaway local reading history
# Keeps the notebook self-contained and offline: writes a few varied reads into a *local* sqlite the
# pipeline will read via `--user`. Does nothing for the golden/history sources. Never touches product data.
if SOURCE == "user":
    os.environ["RWE_DB_URL"] = "sqlite:///%s/data/metric_validation_demo.db" % REPO_ROOT
    os.makedirs(os.path.join(REPO_ROOT, "data"), exist_ok=True)
    import store
    s = store.Store()
    # Use the given USER id if it already exists, else create a stable demo identity (reads have a
    # foreign key to a real user row, so the user must exist before we can add reads).
    if USER.strip().isdigit() and s.get_user(int(USER)) is not None:
        uid = int(USER)
    else:
        uid = s.upsert_user_by_identity("dev", "metric-validation-demo",
                                        "mv@infodiet.local", "Validation Reader").id
    USER = str(uid)
    ARGS = ["--user", USER]                       # rebuild: the id may have been assigned just now
    if s.count_reads(uid) == 0:
        seed = [
            ("https://bbc.com/news/politics-a", {"title": "Senate committee debates the funding bill",
                "category": "Politics", "outlet": "BBC", "lean": -1.2, "political": True,
                "register": "reporting", "emotion": {"fear": .05, "outrage": .05, "analysis": .5, "positive": .2, "neutral": .2}}),
            ("https://foxnews.com/politics-b", {"title": "Governors respond to the new border plan",
                "category": "Politics", "outlet": "Fox News", "lean": 1.4, "political": True,
                "register": "reporting", "emotion": {"fear": .05, "outrage": .1, "analysis": .45, "positive": .2, "neutral": .2}}),
            ("https://reuters.com/business-c", {"title": "Markets steady after earnings week",
                "category": "Business", "outlet": "Reuters", "lean": 0.0, "political": False,
                "register": "reporting", "emotion": {"fear": .05, "outrage": .05, "analysis": .5, "positive": .2, "neutral": .2}}),
            ("https://theverge.com/tech-d", {"title": "A new chip pushes laptops further",
                "category": "Technology", "outlet": "The Verge", "lean": 0.0, "political": False,
                "register": "reporting", "emotion": {"fear": .02, "outrage": .03, "analysis": .55, "positive": .2, "neutral": .2}}),
            ("https://npr.org/health-e", {"title": "Study links sleep to memory",
                "category": "Health", "outlet": "NPR", "lean": -0.5, "political": False,
                "register": "reporting", "emotion": {"fear": .05, "outrage": .05, "analysis": .5, "positive": .2, "neutral": .2}}),
        ]
        for url, scored in seed:
            s.add_read(int(USER), url, scored)
    print("stored user #%s has %d reads (local demo DB)" % (USER, s.count_reads(int(USER))))
else:
    print("skipped (source is not 'user')")

In [ ]:
#@title 3 · Run the Metric Validation Pipeline
import json, subprocess, sys

def run_pipeline_cli(extra=None):
    cmd = [sys.executable, "examples/validate_metrics.py", *ARGS, *(extra or []), "--report", "json"]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if not p.stdout.strip():
        raise RuntimeError("pipeline produced no JSON:\n" + p.stderr)
    return json.loads(p.stdout), p.returncode

RESULT, EXIT = run_pipeline_cli()
S = RESULT["summary"]
print("=" * 64)
print("DATASET  :", RESULT["dataset"], "  ·  readers:", RESULT["population"],
      "  ·  catalog C =", RESULT["catalogCategories"])
print("STATUS   :", "PASS ✅" if RESULT["passed"] else "FAIL ❌", "  (exit code %d)" % EXIT)
print("RAW      : %d/%d equal | DISPLAYED: %d/%d equal | HELPER: %d/%d equal"
      % (S["raw"]["passed"], S["raw"]["total"], S["displayed"]["passed"], S["displayed"]["total"],
         S["helperParity"]["passed"], S["helperParity"]["total"]))
d = RESULT["drift"]
print("DRIFT    :", "baseline" if d.get("baseline") else ("DRIFTED ⚠️" if d.get("drifted") else "no drift"))
print("=" * 64)

# Raw metrics per reader (the deterministic, population-independent layer).
print("\nRAW metrics (independent calculation):")
for reader, metrics in RESULT["independentRaw"].items():
    vals = "  ".join("%s=%s" % (k, ("n/a" if v is None or v != v else round(v, 4))) for k, v in metrics.items())
    print("  %-16s %s" % (reader, vals))

# Displayed / "dashboard" scores (the percentile layer users see), when a population exists.
disp = RESULT["comparison"]["displayed"]
if disp:
    print("\nDISPLAYED (dashboard) percentile scores — production vs independent:")
    print("  %-16s %-16s %10s %12s" % ("metric", "reader", "dashboard", "independent"))
    for r in disp:
        exp = "n/a" if r["expected"] is None or r["expected"] != r["expected"] else round(r["expected"], 1)
        app = "n/a" if r["application"] is None or r["application"] != r["application"] else round(r["application"], 1)
        print("  %-16s %-16s %10s %12s" % (r["metric"], r["reader"], exp, app))
else:
    print("\n(DISPLAYED layer needs a population of >= 2 — use PERSONA='all' to see percentile scores.)")

In [ ]:
#@title 4 · Visualizations (from the pipeline output — no recomputation)
import matplotlib
matplotlib.use("Agg") if not os.environ.get("DISPLAY") and "google.colab" not in sys.modules else None
import matplotlib.pyplot as plt
import numpy as np

summary, raw_rows = RESULT["summary"], RESULT["comparison"]["raw"]
disp_rows = RESULT["comparison"]["displayed"]
graded_raw = [r for r in raw_rows if r["pass"] is not None]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# (1) PASS / FAIL summary per layer.
layers = ["raw", "displayed", "helperParity"]
passed = [summary[l]["passed"] for l in layers]
total  = [summary[l]["total"] for l in layers]
failed = [t - p for p, t in zip(passed, total)]
x = np.arange(len(layers))
axes[0].bar(x, passed, label="equal", color="#55A868")
axes[0].bar(x, failed, bottom=passed, label="mismatch", color="#C44E52")
axes[0].set_xticks(x); axes[0].set_xticklabels(["Raw", "Displayed", "Helper"])
axes[0].set_title("PASS / FAIL by layer"); axes[0].set_ylabel("checks"); axes[0].legend()

# (2) Independent vs production agreement — max |Δ| per metric (log scale; ~0 = exact).
by_metric = {}
for r in graded_raw:
    if r.get("delta") is not None and r["delta"] == r["delta"]:
        by_metric.setdefault(r["metric"], []).append(r["delta"])
if by_metric:
    metrics = list(by_metric); deltas = [max(by_metric[m]) or 1e-18 for m in metrics]
    axes[1].barh(range(len(metrics)), deltas, color="#4C72B0")
    axes[1].set_yticks(range(len(metrics))); axes[1].set_yticklabels(metrics, fontsize=8)
    axes[1].set_xscale("log"); axes[1].set_xlabel("max |expected − independent|")
    axes[1].set_title("Raw agreement (smaller = exact)")
plt.tight_layout(); plt.show()

# (3) Raw value vs displayed (dashboard) score, per metric — how a raw number becomes the 0–100 score.
if disp_rows:
    readers = RESULT["readers"]; pmetrics = sorted({r["metric"] for r in disp_rows})
    dmap = {(r["metric"], r["reader"]): r["application"] for r in disp_rows}
    fig2, ax = plt.subplots(figsize=(12, 4.2))
    width = 0.8 / max(1, len(readers))
    for i, reader in enumerate(readers):
        ys = [(dmap.get((m, reader)) or np.nan) for m in pmetrics]
        ax.bar(np.arange(len(pmetrics)) + i * width, ys, width, label=reader)
    ax.set_xticks(np.arange(len(pmetrics)) + 0.4 - width / 2); ax.set_xticklabels(pmetrics, fontsize=8, rotation=20)
    ax.set_ylabel("displayed percentile (0–100)"); ax.set_title("Displayed (dashboard) scores by persona")
    ax.legend(fontsize=7, ncol=3); plt.tight_layout(); plt.show()
print("charts rendered.")

In [ ]:
#@title 4b · Drift history (record two runs, then chart a metric over time)
# Golden metrics are deterministic, so drift over identical runs is zero. To *demonstrate* the signal,
# we record a baseline then a deliberately perturbed run under the same dataset name and plot both.
import tempfile, json, subprocess, sys, os

HIST = os.path.join(tempfile.gettempdir(), "mv_drift_demo.jsonl")
RJSON = os.path.join(tempfile.gettempdir(), "mv_drift_reads.json")
open(HIST, "w").close()

def _reads(reporting_second):
    reg = "reporting" if reporting_second else "opinion"
    return {"reads": [
        {"scored": {"category": "Politics", "outlet": "BBC", "lean": -0.9, "political": True,
                    "register": "reporting", "emotion": {"fear": .1, "outrage": .1, "analysis": .5, "positive": .2, "neutral": .1}}},
        {"scored": {"category": "Business", "outlet": "Reuters", "lean": 0.0, "political": False, "register": reg}},
    ]}

series = []
for label, second in [("baseline", True), ("perturbed", False)]:
    json.dump(_reads(second), open(RJSON, "w"))
    p = subprocess.run([sys.executable, "examples/validate_metrics.py", "--history", RJSON,
                        "--record", "--history-file", HIST, "--report", "json"], capture_output=True, text=True)
    r = json.loads(p.stdout)
    val = list(r["independentRaw"].values())[0]["reportingRatio"]
    series.append((label, val))
    print("%-10s reportingRatio = %s  ·  drift: %s"
          % (label, round(val, 3), "baseline" if r["drift"].get("baseline") else
             ("DRIFTED" if r["drift"]["drifted"] else "none")))

import matplotlib.pyplot as plt
labels, vals = zip(*series)
plt.figure(figsize=(6, 3.6))
plt.plot(range(len(vals)), vals, "-o", color="#C44E52")
plt.xticks(range(len(labels)), labels); plt.ylabel("reportingRatio (raw)")
plt.title("Drift history — a metric across recorded runs"); plt.ylim(-0.05, 1.05)
plt.tight_layout(); plt.show()

In [ ]:
#@title 5 · Validation report (Raw · Displayed · Drift · Summary · Overall Status)
import subprocess, sys
p = subprocess.run([sys.executable, "examples/validate_metrics.py", *ARGS, "--report", "text"],
                   capture_output=True, text=True)
print(p.stdout)

In [ ]:
#@title 6 · Export (JSON · CSV · HTML — from existing output)
import json, pandas as pd, os

rows = []
for layer in ("raw", "displayed", "helperParity"):
    for r in RESULT["comparison"][layer]:
        rows.append({"layer": layer, "metric": r.get("metric"), "reader": r.get("reader"),
                     "expected": r.get("expected"), "application": r.get("application"),
                     "delta": r.get("delta"), "pass": r.get("pass"), "basis": r.get("basis")})
df = pd.DataFrame(rows)

json_path = "metric_validation.json"; csv_path = "metric_validation.csv"; html_path = "metric_validation.html"
with open(json_path, "w") as f:
    json.dump(RESULT, f, indent=2)
df.to_csv(csv_path, index=False)
status = "PASS" if RESULT["passed"] else "FAIL"
html = ("<h2>Metric Validation — %s (%s)</h2>" % (RESULT["dataset"], status)) + df.to_html(index=False)
with open(html_path, "w") as f:
    f.write(html)
print("wrote:", json_path, csv_path, html_path)
try:
    from google.colab import files          # offer downloads in Colab
    for pth in (json_path, csv_path, html_path):
        files.download(pth)
except Exception:
    print("(in Colab, these download automatically; locally they are in %s)" % os.getcwd())
df.head(12)

## 7 · (optional) Dashboard Verification — compare vs a running application

**Off by default (fully offline above).** If you have the product engine running (e.g. via the
[product notebook](information_health_colab.ipynb) or `uvicorn`), point `DASHBOARD_BASE_URL` at it to
verify that the numbers the **Dashboard/API shows users** match the independent calculation.

What is compared, honestly:
- **Raw, population-independent fields** the report exposes — `attention` (emotion mix), `topics`, and
  `sources` shares — should match the independent calculation **exactly** (they depend only on your
  reads). `viewpoint` (left/centre/right) matches too unless the engine confidence-weighted it.
- **Displayed percentile scores** are ranked against the *live* population (the augmented corpus), a
  different population than the offline golden set, so their absolute values are shown for context, not
  asserted equal. The **offline pipeline remains the authoritative PASS/FAIL gate.**

In [ ]:
#@title 7 · Dashboard Verification (needs a running engine; leave URL blank to skip)
import json, os, sys, urllib.request

DASHBOARD_BASE_URL = ""  #@param {type:"string"}     # e.g. http://127.0.0.1:8000  (blank = skip, offline)
DASH_USER = "1"          #@param {type:"string"}

def _req(method, path, body=None, headers=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(DASHBOARD_BASE_URL + path, data=data, method=method,
                                 headers={"Content-Type": "application/json", **(headers or {})})
    with urllib.request.urlopen(req, timeout=20) as r:
        return json.loads(r.read().decode())

if not DASHBOARD_BASE_URL.strip():
    print("skipped — offline default. Set DASHBOARD_BASE_URL to a running engine to verify the dashboard.")
else:
    hdr = {"X-IH-User-Id": str(DASH_USER)}
    try:
        history = _req("GET", "/api/me/history", headers=hdr)
    except Exception:
        history = []
    if not history:                                   # self-contained: seed a demo reader if empty
        uid = _req("POST", "/api/internal/users", {"provider": "dev",
                   "providerAccountId": "mv@infodiet.local", "email": "mv@infodiet.local",
                   "displayName": "Validation Reader"})["userId"]
        hdr = {"X-IH-User-Id": str(uid)}; DASH_USER = str(uid)
        sample = [("https://bbc.com/p1", "Senate committee debates the funding bill"),
                  ("https://foxnews.com/p2", "Governors clash over the new border plan"),
                  ("https://reuters.com/b1", "Markets steady after a strong earnings week"),
                  ("https://theverge.com/t1", "A new chip pushes laptops further"),
                  ("https://npr.org/h1", "Study links better sleep to memory"),
                  ("https://apnews.com/w1", "Aid convoy reaches the region"),
                  ("https://theguardian.com/c1", "A festival reshapes the old quarter"),
                  ("https://wsj.com/o1", "Opinion: rethinking the tax plan")]
        _req("POST", "/api/me/reads", {"reads": [{"url": u, "title": t} for u, t in sample]}, hdr)
        history = _req("GET", "/api/me/history", headers=hdr)

    report = _req("GET", "/api/report", headers=hdr)              # what the dashboard shows this user

    # Reconstruct scored reads from the reading history the API exposes (Article shape), then compute
    # the independent raw intermediates with the pipeline's Normalize + Feature stages.
    reads = []
    for e in history:
        a = e.get("article", {}); topic = a.get("topic", "") or ""
        reads.append({"scored": {"category": topic, "outlet": a.get("outlet", ""),
                                 "lean": a.get("lean"), "political": "politic" in topic.lower(),
                                 "register": a.get("register"), "emotion": a.get("emotion"),
                                 "title": a.get("headline", "")}})
    from metric_pipeline import normalize as _norm, engine as _eng
    feat = _eng.features(_norm.normalize(reads))

    def _cmp(name, dash, indep):
        ok = dash is not None and indep is not None and abs(float(dash) - float(indep)) <= 1e-6
        print("  %-22s dashboard=%-9s independent=%-9s  %s"
              % (name, round(dash, 4) if dash is not None else "n/a",
                 round(indep, 4) if indep is not None else "n/a", "MATCH ✅" if ok else "differs"))

    print("Dashboard Verification — user #%s · report mode: %s · %d reads"
          % (DASH_USER, report.get("mode"), len(reads)))
    print("\nAttention profile (emotion mix) — population-independent, verified to match exactly:")
    att, fe = report.get("attention", {}), feat["emotion_means"]
    allmatch = True
    for b in ("fear", "outrage", "analysis", "positive", "neutral"):
        _cmp("attention:" + b, att.get(b), fe.get(b))
        allmatch = allmatch and att.get(b) is not None and abs(float(att[b]) - float(fe.get(b, 0))) <= 1e-6
    print("  → attention profile the dashboard shows " + ("MATCHES the independent calculation ✅"
          if allmatch else "differs — inspect above"))

    # The rest is shown for reference, not asserted: topic/source parity needs fully-scored reads
    # (the history projection can omit category on some sources), the political flag isn't exposed by
    # history, and the displayed percentiles rank vs the LIVE population (a different population than
    # the offline golden set), so their absolute values are expected to differ.
    print("\nDashboard context (not asserted — see notes above):")
    vp = report.get("viewpoint", {})
    print("  viewpoint mix:  left=%s center=%s right=%s" % (vp.get("left"), vp.get("center"), vp.get("right")))
    print("  top topics:     " + ", ".join("%s %.0f%%" % (t["topic"], 100 * t["share"])
                                            for t in report.get("topics", [])[:4]))
    print("  displayed scores the dashboard renders:")
    for m in report.get("metrics", []):
        print("    %-18s %s/100" % (m.get("key"), m.get("score")))
    print("\nThe offline pipeline above remains the authoritative PASS/FAIL gate.")